In [1]:
from environment import *
from movement import *


import numpy as np
import pandas as pd

from tqdm import tqdm

import pickle
import math
import visual

pygame 2.1.0 (SDL 2.0.16, Python 3.10.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
with open("Q.pkl", "rb") as f:
    Q = pickle.load(f)

In [2]:
def get_actions(combined_action):
    action1 = math.floor(combined_action / 4)
    action2 = combined_action % 4
    return All_Actions(action1, action2)


In [3]:
Q = {}

all_positions = []
all_rewards = []
env = GridWorld_Portal()

Q[env.all_entity_positions] = [200] * 16

In [4]:
def epsilon_greedy(state, epsilon):

    if np.random.random() < epsilon:
        return np.random.randint(12)
    else:
        combined_a = np.argmax(Q[state]).item()
        return combined_a

In [5]:
def update_Q(previous_state, action, next_state, reward):

    if next_state not in Q:
        Q[next_state] = [200] * 16

    Q[previous_state][action] = Q[previous_state][action] + 0.1 * (
        reward + 0.99 * max(Q[next_state]) - Q[previous_state][action]
    )

In [6]:
def get_reward(previous_state, next_state):

    reward = 0
    _, _, box_pos = previous_state.get_positions()
    next_pos1, next_pos2, next_box_pos = next_state.get_positions()

    if next_state not in Q:
        reward += 1


    if box_pos != next_box_pos:
        reward += 2

    if next_pos1.get_position()[0] > 9:
        reward += 3

    if next_pos2.get_position()[0] > 9:
        reward += 3

    return reward

In [7]:
for i in tqdm(range(30000)):

    env.reset()
    rewards = [0] * 1000
    positions = [0] * 1000

    for j in range(1000):
        previous_state = env.all_entity_positions
        positions[j] = previous_state

        combined_action = epsilon_greedy(previous_state, 0.2)
        actions = get_actions(combined_action)

        next_state = env.step(actions)

        reward = get_reward(previous_state, next_state)

        rewards[j] = reward

        update_Q(previous_state, combined_action, next_state, reward)



    all_positions.append(positions)
    all_rewards.append(rewards)


100%|██████████| 30000/30000 [21:53<00:00, 22.85it/s]  


In [12]:
positions = all_positions[-20]

In [15]:
positions = [position.get_positions() for position in positions]

In [16]:
positions

[(Position at 0, 2, Position at 3, 5, Position at 4, 2),
 (Position at 0, 1, Position at 4, 5, Position at 4, 2),
 (Position at 0, 0, Position at 4, 5, Position at 4, 2),
 (Position at 1, 0, Position at 4, 4, Position at 4, 2),
 (Position at 2, 0, Position at 3, 4, Position at 4, 2),
 (Position at 3, 0, Position at 4, 4, Position at 4, 2),
 (Position at 3, 1, Position at 4, 4, Position at 4, 2),
 (Position at 3, 0, Position at 3, 4, Position at 4, 2),
 (Position at 3, 1, Position at 3, 3, Position at 4, 2),
 (Position at 3, 2, Position at 3, 2, Position at 4, 2),
 (Position at 4, 2, Position at 4, 2, Position at 5, 2),
 (Position at 5, 2, Position at 5, 2, Position at 6, 2),
 (Position at 5, 1, Position at 5, 1, Position at 6, 2),
 (Position at 6, 1, Position at 6, 1, Position at 6, 2),
 (Position at 6, 2, Position at 6, 2, Position at 6, 3),
 (Position at 6, 1, Position at 6, 1, Position at 6, 3),
 (Position at 6, 2, Position at 6, 1, Position at 6, 3),
 (Position at 6, 2, Position at

In [19]:
visual.player1_positions = [position[0].get_position() for position in positions]
visual.player2_positions = [position[1].get_position() for position in positions]
visual.movable_object = [position[2].get_position() for position in positions]

In [21]:
game = visual.Game()
game.game_loop()

In [ ]:
# Video().build_video("office_and_garden_output14", game.frames)

In [76]:
def show_value_counts(all_rewards):
    for rewards in all_rewards:
        print(pd.Series(rewards).value_counts())

In [129]:
# Save
with open("Q.pkl", "wb") as f:
    pickle.dump(Q, f)